# Rule Sensitivity Analysis

Purpose: measure how much IoU/Dice changes when **one threshold changes while all other thresholds stay fixed at the same baseline configuration**.

This notebook does **not select a best value, does not update the baseline after each sweep, and does not create a recommended-rule CSV**.

It writes one CSV per parameter plus a combined `rule_sensitivity_all.csv`.


In [1]:
from pathlib import Path
from dataclasses import replace
from itertools import product
import random
import sys

import cv2
import numpy as np
import pandas as pd


def find_project_root(start: Path = Path.cwd()) -> Path:
    for candidate in [start, *start.parents]:
        if (
            (candidate / "banana_quality" / "config.py").exists()
            and (candidate / "data" / "annotations").exists()
        ):
            return candidate
    raise FileNotFoundError(
        "Could not find project root. Start Jupyter from inside the banana project."
    )

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from banana_quality.config import ProcessingConfig
from banana_quality.segmentation.kmeans import prepare_image
from banana_quality.segmentation.mask_refinement import banana_mask_from_prepared
from banana_quality.blemish.detection import blemish_mask_from_prepared
from banana_quality.surface_quality import blemish_percentage
from banana_quality.morphology import _morphology, _remove_tiny_components

DATASET_ROOT = PROJECT_ROOT / "data" / "annotations"
RESULTS_DIR = PROJECT_ROOT / "evaluation" / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
SPLIT_SEED = 42
FINAL_EVALUATION_COUNT = 9

print("Project root:", PROJECT_ROOT)
print("Dataset root:", DATASET_ROOT)


Project root: c:\Users\enton\Project\python\banana_quality_ripeness_assessment_system
Dataset root: c:\Users\enton\Project\python\banana_quality_ripeness_assessment_system\data\annotations


In [2]:
images_dir = DATASET_ROOT / "images"
image_paths = sorted(
    p for p in images_dir.iterdir()
    if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
)

shuffled_paths = image_paths.copy()
rng = random.Random(SPLIT_SEED)
rng.shuffle(shuffled_paths)
final_names = {p.name for p in shuffled_paths[:FINAL_EVALUATION_COUNT]}

split_df = pd.DataFrame([
    {
        "image": p.name,
        "split": "final_evaluation" if p.name in final_names else "tuning",
    }
    for p in image_paths
]).sort_values(["split", "image"]).reset_index(drop=True)

split_df.to_csv(RESULTS_DIR / "dataset_split.csv", index=False)

assert len(image_paths) == 30, f"Expected 30 images, found {len(image_paths)}"
assert int((split_df["split"] == "tuning").sum()) == 21
assert int((split_df["split"] == "final_evaluation").sum()) == 9

print(split_df["split"].value_counts())


split
tuning              21
final_evaluation     9
Name: count, dtype: int64


In [3]:
def matching_mask(folder: Path, image_path: Path) -> Path:
    path = folder / f"{image_path.stem}.png"
    if not path.exists():
        raise FileNotFoundError(path)
    return path


def load_binary_mask(path: Path, target_shape: tuple[int, int]) -> np.ndarray:
    mask = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if mask is None:
        raise ValueError(f"Cannot read mask: {path}")
    mask = np.where(mask > 0, 255, 0).astype(np.uint8)
    if mask.shape != target_shape:
        h, w = target_shape
        mask = cv2.resize(mask, (w, h), interpolation=cv2.INTER_NEAREST)
    return mask


def resize_binary_mask(mask: np.ndarray, target_shape: tuple[int, int]) -> np.ndarray:
    h, w = target_shape
    resized = cv2.resize(
        np.asarray(mask, dtype=np.uint8),
        (w, h),
        interpolation=cv2.INTER_NEAREST,
    )
    return np.where(resized > 0, 255, 0).astype(np.uint8)


def iou_score(prediction: np.ndarray, ground_truth: np.ndarray) -> float:
    pred = prediction > 0
    gt = ground_truth > 0
    union = int(np.logical_or(pred, gt).sum())
    if union == 0:
        return 1.0
    return float(np.logical_and(pred, gt).sum() / union)


def dice_score(prediction: np.ndarray, ground_truth: np.ndarray) -> float:
    pred = prediction > 0
    gt = ground_truth > 0
    denominator = int(pred.sum() + gt.sum())
    if denominator == 0:
        return 1.0
    return float(2.0 * np.logical_and(pred, gt).sum() / denominator)


def load_samples(split_name: str):
    names = set(split_df.loc[split_df["split"] == split_name, "image"])
    samples = []
    for image_path in image_paths:
        if image_path.name not in names:
            continue
        image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
        if image is None:
            raise ValueError(f"Cannot read image: {image_path}")
        shape = image.shape[:2]
        banana_gt = load_binary_mask(
            matching_mask(DATASET_ROOT / "banana_masks", image_path),
            shape,
        )
        blemish_gt = load_binary_mask(
            matching_mask(DATASET_ROOT / "blemish_masks", image_path),
            shape,
        )
        samples.append({
            "name": image_path.name,
            "image": image,
            "shape": shape,
            "banana_gt": banana_gt,
            "blemish_gt": blemish_gt,
        })
    return samples


def clean_variant(raw_mask, banana_mask, opening, closing, min_area_ratio):
    raw = np.where(np.asarray(raw_mask) > 0, 255, 0).astype(np.uint8)
    opened = _morphology(raw, cv2.MORPH_OPEN, int(opening))
    closed = _morphology(opened, cv2.MORPH_CLOSE, int(closing))
    minimum_area = max(
        8,
        int((np.asarray(banana_mask) > 0).sum() * float(min_area_ratio)),
    )
    final_mask = _remove_tiny_components(closed, minimum_area)
    final_mask[np.asarray(banana_mask) == 0] = 0
    return final_mask


def variant_mask_from_steps(
    steps,
    banana_mask,
    use_rule_a,
    use_rule_b,
    opening,
    closing,
    min_area_ratio,
):
    candidate = np.asarray(steps["kmeans_region_candidate"]) > 0
    if use_rule_a:
        candidate |= np.asarray(steps["rule_a_mask"]) > 0
    if use_rule_b:
        candidate |= np.asarray(steps["rule_b_mask"]) > 0
    return clean_variant(
        np.where(candidate, 255, 0).astype(np.uint8),
        banana_mask,
        opening,
        closing,
        min_area_ratio,
    )


## 1. Load tuned core parameters and set the fixed sensitivity baseline

Core parameters come automatically from Notebook 01. You still choose the retained Rule A/B structure from Notebook 02 and set the baseline rule thresholds below.

**Important:** sensitivity analysis does not select a new recommended threshold. Each tested parameter is varied from the same fixed baseline while all other values remain unchanged.


In [4]:
core_path = RESULTS_DIR / "recommended_core_parameters.csv"
if not core_path.exists():
    raise FileNotFoundError(
        "Run 01_parameter_tuning.ipynb first. Missing: " + str(core_path)
    )

core = pd.read_csv(core_path).iloc[0]

BANANA_K = int(core["kmeans_k"])
BANANA_ACCEPTANCE = float(core["banana_cluster_score_threshold"])
BANANA_OPENING = int(core["banana_opening_kernel"])
BANANA_CLOSING = int(core["banana_closing_kernel"])
BLEMISH_K = int(core["blemish_kmeans_k"])
BLEMISH_CHROMA = float(core["blemish_cluster_chroma_threshold"])
BLEMISH_OPENING = int(core["blemish_opening_kernel"])
BLEMISH_CLOSING = int(core["blemish_closing_kernel"])

print("Loaded tuned core parameters from Notebook 01:")
display(pd.DataFrame([{
    "banana_k": BANANA_K,
    "banana_acceptance": BANANA_ACCEPTANCE,
    "banana_opening": BANANA_OPENING,
    "banana_closing": BANANA_CLOSING,
    "blemish_k": BLEMISH_K,
    "blemish_chroma": BLEMISH_CHROMA,
    "blemish_opening": BLEMISH_OPENING,
    "blemish_closing": BLEMISH_CLOSING,
}]))

# Rule choice: set this manually after inspecting Notebook 02.
USE_RULE_A = True
USE_RULE_B = True

# Fixed BASELINE thresholds for sensitivity analysis.
# These values are NEVER automatically replaced by the best sensitivity row.
BASE_HUE_DIFFERENCE = 7.0
BASE_DARKNESS = 32.0
BASE_STRONG_HUE_DIFFERENCE = 10.0
BASE_REDNESS = 11.0
BASE_YELLOW_LOSS = 20.0
BASE_MIN_SATURATION = 35.0
BASE_HIGHLIGHT_VALUE = 220.0
BASE_HIGHLIGHT_SATURATION = 100.0

DEFAULTS = ProcessingConfig()
baseline_config = replace(
    DEFAULTS,
    kmeans_k=int(BANANA_K),
    banana_cluster_score_threshold=float(BANANA_ACCEPTANCE),
    blemish_kmeans_k=int(BLEMISH_K),
    blemish_cluster_chroma_threshold=float(BLEMISH_CHROMA),
    hue_drop_threshold=float(BASE_HUE_DIFFERENCE),
    darkness_threshold=float(BASE_DARKNESS),
    strong_hue_drop_threshold=float(BASE_STRONG_HUE_DIFFERENCE),
    redness_threshold=float(BASE_REDNESS),
    yellow_loss_threshold=float(BASE_YELLOW_LOSS),
    blemish_min_saturation=float(BASE_MIN_SATURATION),
    highlight_value_threshold=float(BASE_HIGHLIGHT_VALUE),
    highlight_saturation_threshold=float(BASE_HIGHLIGHT_SATURATION),
)


Loaded tuned core parameters from Notebook 01:


,banana_k,banana_acceptance,banana_opening,banana_closing,blemish_k,blemish_chroma,blemish_opening,blemish_closing
0,7,0.35,3,7,6,21.0,3,3


## 2. Prepare the 21 tuning images


In [5]:
tuning_samples = load_samples("tuning")
cache = []
for sample in tuning_samples:
    prepared = prepare_image(sample["image"], baseline_config)
    banana_steps = banana_mask_from_prepared(
        prepared,
        opening_kernel_size=int(BANANA_OPENING),
        closing_kernel_size=int(BANANA_CLOSING),
    )
    banana_working = np.asarray(banana_steps["banana_mask"], dtype=np.uint8)
    banana_pred = resize_binary_mask(banana_working, sample["shape"])
    cache.append({**sample, "prepared": prepared, "banana_working": banana_working, "banana_pred": banana_pred})
print("Prepared:", len(cache))


Prepared: 21


## 3. Sensitivity helper


In [6]:
def evaluate_config(config):
    ious, dices, pct_errors = [], [], []
    for sample in cache:
        steps = blemish_mask_from_prepared(
            sample["prepared"],
            sample["banana_working"],
            opening_kernel_size=int(BLEMISH_OPENING),
            closing_kernel_size=int(BLEMISH_CLOSING),
            config=config,
        )
        working = variant_mask_from_steps(
            steps,
            sample["banana_working"],
            use_rule_a=USE_RULE_A,
            use_rule_b=USE_RULE_B,
            opening=BLEMISH_OPENING,
            closing=BLEMISH_CLOSING,
            min_area_ratio=config.min_blemish_area_ratio,
        )
        pred = resize_binary_mask(working, sample["shape"])
        ious.append(iou_score(pred, sample["blemish_gt"]))
        dices.append(dice_score(pred, sample["blemish_gt"]))
        pred_pct = blemish_percentage(sample["banana_pred"], pred)
        gt_pct = blemish_percentage(sample["banana_gt"], sample["blemish_gt"])
        pct_errors.append(abs(pred_pct - gt_pct))
    return {
        "blemish_mean_iou": float(np.mean(ious)),
        "blemish_mean_dice": float(np.mean(dices)),
        "blemish_percentage_mae": float(np.mean(pct_errors)),
    }


def run_sensitivity(parameter_name, values):
    rows = []
    for value in values:
        # IMPORTANT: always start from the SAME baseline_config.
        test_config = replace(baseline_config, **{parameter_name: value})
        rows.append({
            "parameter": parameter_name,
            "value": value,
            **evaluate_config(test_config),
        })
    result = pd.DataFrame(rows)
    filename = f"sensitivity_{parameter_name}.csv"
    result.to_csv(RESULTS_DIR / filename, index=False)
    display(result)
    print("Saved", filename)
    return result


## 4. Run one-at-a-time sensitivity sweeps


In [7]:
all_results = []

if USE_RULE_A:
    all_results.append(run_sensitivity("hue_drop_threshold", [3.0, 5.0, 7.0, 9.0, 11.0]))
    all_results.append(run_sensitivity("darkness_threshold", [20.0, 24.0, 28.0, 32.0, 36.0, 40.0]))

if USE_RULE_B:
    all_results.append(run_sensitivity("strong_hue_drop_threshold", [6.0, 8.0, 10.0, 12.0, 14.0]))
    all_results.append(run_sensitivity("redness_threshold", [7.0, 9.0, 11.0, 13.0, 15.0]))
    all_results.append(run_sensitivity("yellow_loss_threshold", [12.0, 16.0, 20.0, 24.0, 28.0]))

if USE_RULE_A or USE_RULE_B:
    all_results.append(run_sensitivity("blemish_min_saturation", [25.0, 35.0, 45.0]))
    all_results.append(run_sensitivity("highlight_value_threshold", [200.0, 220.0, 240.0]))
    all_results.append(run_sensitivity("highlight_saturation_threshold", [80.0, 100.0, 120.0]))

if all_results:
    sensitivity_all = pd.concat(all_results, ignore_index=True)
else:
    sensitivity_all = pd.DataFrame(columns=[
        "parameter", "value", "blemish_mean_iou", "blemish_mean_dice", "blemish_percentage_mae"
    ])

sensitivity_all.to_csv(RESULTS_DIR / "rule_sensitivity_all.csv", index=False)
print("Saved rule_sensitivity_all.csv")


,parameter,value,blemish_mean_iou,blemish_mean_dice,blemish_percentage_mae
0,hue_drop_threshold,3.0,0.329946,0.446826,8.828684
1,hue_drop_threshold,5.0,0.384686,0.502143,7.826619
2,hue_drop_threshold,7.0,0.385197,0.495743,7.614507
3,hue_drop_threshold,9.0,0.382313,0.489716,7.599044
4,hue_drop_threshold,11.0,0.377770,0.483608,7.663930


Saved sensitivity_hue_drop_threshold.csv


,parameter,value,blemish_mean_iou,blemish_mean_dice,blemish_percentage_mae
0,darkness_threshold,20.0,0.382928,0.493973,7.598825
1,darkness_threshold,24.0,0.383651,0.494579,7.599142
2,darkness_threshold,28.0,0.384185,0.494987,7.610728
3,darkness_threshold,32.0,0.385197,0.495743,7.614507
4,darkness_threshold,36.0,0.386514,0.496670,7.615392
5,darkness_threshold,40.0,0.386652,0.496681,7.572429


Saved sensitivity_darkness_threshold.csv


,parameter,value,blemish_mean_iou,blemish_mean_dice,blemish_percentage_mae
0,strong_hue_drop_threshold,6.0,0.338075,0.451856,7.932142
1,strong_hue_drop_threshold,8.0,0.384951,0.495515,7.610210
2,strong_hue_drop_threshold,10.0,0.385197,0.495743,7.614507
3,strong_hue_drop_threshold,12.0,0.383390,0.494254,7.602280
4,strong_hue_drop_threshold,14.0,0.378610,0.489802,7.600560


...

Saved sensitivity_blemish_min_saturation.csv


,parameter,value,blemish_mean_iou,blemish_mean_dice,blemish_percentage_mae
0,highlight_value_threshold,200.0,0.384365,0.495123,7.572603
1,highlight_value_threshold,220.0,0.385197,0.495743,7.614507
2,highlight_value_threshold,240.0,0.383998,0.494622,7.844361


Saved sensitivity_highlight_value_threshold.csv


,parameter,value,blemish_mean_iou,blemish_mean_dice,blemish_percentage_mae
0,highlight_saturation_threshold,80.0,0.383438,0.494454,7.756171
1,highlight_saturation_threshold,100.0,0.385197,0.495743,7.614507
2,highlight_saturation_threshold,120.0,0.385875,0.496110,7.553433


Saved sensitivity_highlight_saturation_threshold.csv
Saved rule_sensitivity_all.csv


## End of Notebook 03

The generated sensitivity CSVs show **value → mean IoU / mean Dice / percentage MAE** only. No recommended rule threshold is created. Inspect the results for a stable/robust range and make the final choice yourself before Notebook 04.
